# 🎙️ HooperTTS + Qwen3-TTS

An open-source narration compiler for expressive AI voice generation.

Supported

• Voice Cloning
• Gaming News
• Documentary
• YouTube Shorts
• Podcast

GitHub:
https://github.com/XADITYAM/HooperTTS

First Run

The first execution downloads the Qwen3-TTS model (~4.5 GB).

This usually takes 2–5 minutes.

Subsequent generations in the same Colab session are much faster.

In [ ]:
import os

if not os.path.exists("/content/HooperTTS"):
    !git clone https://github.com/guideofoai-blip/HooperTTS-custom.git

%cd /content/HooperTTS

In [ ]:
!pip install -e .

In [ ]:
%cd /content

if not os.path.exists("/content/Qwen3-TTS"):
    !git clone https://github.com/QwenLM/Qwen3-TTS.git

%cd /content/Qwen3-TTS

In [ ]:
!pip install -e .

!pip install \
    faster-whisper==1.1.1 \
    ctranslate2==4.5.0 \
    sentencex \
    pysrt \
    soundfile

In [ ]:
# --- Permanent fix for the Xet/CAS 403 SignatureError ---
# Must run BEFORE anything imports huggingface_hub (qwen_tts does this internally).
import os

!pip uninstall -y hf-xet -q

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

print("Xet disabled, plain HTTP downloads forced.")

In [ ]:
import torch

print("="*40)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

print(torch.__version__)
print("="*40)

In [ ]:
import qwen_tts

print("✓ qwen_tts imported successfully")

In [ ]:
import time
from huggingface_hub import snapshot_download

def robust_snapshot_download(repo_id, max_retries=5, backoff_seconds=10, **kwargs):
    """Retries snapshot_download with backoff to ride out transient HF CDN errors
    (e.g. 403 SignatureError from the xet-bridge CDN, connection resets, etc.).
    """
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return snapshot_download(repo_id=repo_id, **kwargs)
        except Exception as e:
            last_err = e
            wait = backoff_seconds * attempt
            print(f"[attempt {attempt}/{max_retries}] download failed: {e}")
            if attempt < max_retries:
                print(f"Retrying in {wait}s...")
                time.sleep(wait)
    raise last_err

model_path = robust_snapshot_download(
    repo_id="Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    max_workers=1,
)

print(model_path)

In [ ]:
!hoopertts doctor

In [ ]:


%cd /content/HooperTTS

!python app.py